In [ ]:
from importlib.util import find_spec
from pathlib import Path
import json
import os
import re

if find_spec("pandas") is None:
    raise ModuleNotFoundError("Thiếu dependency để build KB chunks bằng notebook chính thức: pandas (pip install pandas)")

import pandas as pd

def find_ai_lab_root(start: Path) -> Path:
    current = start.resolve()
    for candidate in [current] + list(current.parents):
        if candidate.name == "ai_lab":
            return candidate
    raise RuntimeError("Không tìm thấy thư mục ai_lab.")

AI_LAB_ROOT = find_ai_lab_root(Path.cwd())

KB_VERSION = os.getenv("HOMELAB_KB_VERSION", "v1")
RETRIEVER_VERSION = os.getenv("HOMELAB_RETRIEVER_VERSION", KB_VERSION)
REPORT_VERSION = os.getenv("HOMELAB_REPORT_VERSION", KB_VERSION)

DATASETS_DIR = AI_LAB_ROOT / "datasets"
ARTIFACTS_DIR = AI_LAB_ROOT / "artifacts" / f"retriever_{RETRIEVER_VERSION}"
REPORTS_DIR = AI_LAB_ROOT / "reports"

DATASETS_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

MEDICAL_KB_PATH = DATASETS_DIR / f"medical_kb_{KB_VERSION}.json"
KB_CHUNKS_PATH = ARTIFACTS_DIR / f"kb_chunks_{KB_VERSION}.json"
CHUNK_METADATA_PATH = ARTIFACTS_DIR / "chunk_metadata.json"
CHUNK_STATS_PATH = REPORTS_DIR / f"kb_chunk_stats_{REPORT_VERSION}.csv"

print("AI_LAB_ROOT =", AI_LAB_ROOT)
print("MEDICAL_KB_PATH =", MEDICAL_KB_PATH)


In [ ]:
with open(MEDICAL_KB_PATH, "r", encoding="utf-8") as f:
    medical_kb = json.load(f)

print("Số knowledge items:", len(medical_kb))
pd.DataFrame(medical_kb)[["id", "source_id", "title", "section", "risk_level"]]

In [ ]:
required_fields = [
    "id", "doc_id", "source_id", "source_name", "source_url",
    "section", "title", "content", "risk_level",
    "tags", "keywords", "test_types",
    "faq_type", "safety_notes", "review_status", "use_in_v1",
    "language", "locale"
]

errors = []

for idx, item in enumerate(medical_kb):
    for field in required_fields:
        if field not in item:
            errors.append({"row": idx, "id": item.get("id"), "missing_field": field})

df_errors = pd.DataFrame(errors)
print("Số lỗi schema:", len(df_errors))
df_errors.head()

In [ ]:
def clean_text(text: str) -> str:
    if not text:
        return ""
    text = text.replace("\u00a0", " ")
    text = re.sub(r"\r\n?", "\n", text)
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()

def ensure_list(value):
    if value is None:
        return []
    if isinstance(value, list):
        return value
    if isinstance(value, str):
        value = value.strip()
        if not value:
            return []
        return [x.strip() for x in value.split(",") if x.strip()]
    return []

In [ ]:
def build_chunk_text(item):
    parts = [
        f"Tiêu đề: {clean_text(item['title'])}",
        f"Mục: {item['section']}",
        f"Mức độ rủi ro: {item['risk_level']}"
    ]

    keywords = ensure_list(item.get("keywords"))
    tags = ensure_list(item.get("tags"))
    test_types = ensure_list(item.get("test_types"))

    if keywords:
        parts.append(f"Từ khóa: {', '.join(keywords)}")
    if tags:
        parts.append(f"Nhãn: {', '.join(tags)}")
    if test_types:
        parts.append(f"Loại xét nghiệm: {', '.join(test_types)}")
    if item.get("faq_type"):
        parts.append(f"Loại FAQ: {item['faq_type']}")
    if item.get("safety_notes"):
        parts.append(f"Lưu ý an toàn: {clean_text(item['safety_notes'])}")

    parts.append(f"Nội dung: {clean_text(item['content'])}")

    return "\n".join(parts)

In [ ]:
kb_chunks = []
chunk_metadata = []

for item in medical_kb:
    chunk = {
        "chunk_id": f"{item['id']}_c1",
        "kb_id": item["id"],
        "doc_id": item["doc_id"],
        "source_id": item["source_id"],
        "source_name": item["source_name"],
        "source_url": item["source_url"],
        "section": item["section"],
        "title": clean_text(item["title"]),
        "content": clean_text(item["content"]),
        "chunk_text": build_chunk_text(item),
        "risk_level": item["risk_level"],
        "tags": ensure_list(item.get("tags")),
        "keywords": ensure_list(item.get("keywords")),
        "test_types": ensure_list(item.get("test_types")),
        "faq_type": item.get("faq_type", ""),
        "safety_notes": clean_text(item.get("safety_notes", "")),
        "review_status": item.get("review_status", "pending"),
        "use_in_v1": bool(item.get("use_in_v1", False)),
        "language": item.get("language", "vi"),
        "locale": item.get("locale", "vi-VN")
    }

    kb_chunks.append(chunk)

    chunk_metadata.append({
        "chunk_id": chunk["chunk_id"],
        "kb_id": chunk["kb_id"],
        "source_id": chunk["source_id"],
        "source_name": chunk["source_name"],
        "section": chunk["section"],
        "title": chunk["title"],
        "risk_level": chunk["risk_level"],
        "faq_type": chunk["faq_type"],
        "use_in_v1": chunk["use_in_v1"]
    })

print("Số chunks tạo ra:", len(kb_chunks))
pd.DataFrame(kb_chunks)[["chunk_id", "title", "section", "risk_level"]]

In [ ]:
stats_rows = []

for chunk in kb_chunks:
    stats_rows.append({
        "chunk_id": chunk["chunk_id"],
        "source_id": chunk["source_id"],
        "section": chunk["section"],
        "risk_level": chunk["risk_level"],
        "title_len": len(chunk["title"]),
        "content_len": len(chunk["content"]),
        "chunk_text_len": len(chunk["chunk_text"]),
        "keyword_count": len(chunk["keywords"]),
        "tag_count": len(chunk["tags"])
    })

df_stats = pd.DataFrame(stats_rows)
df_stats.to_csv(CHUNK_STATS_PATH, index=False, encoding="utf-8-sig")

print("Đã ghi:", CHUNK_STATS_PATH)
df_stats.describe(include="all")

In [ ]:
with open(KB_CHUNKS_PATH, "w", encoding="utf-8") as f:
    json.dump(kb_chunks, f, ensure_ascii=False, indent=2)

with open(CHUNK_METADATA_PATH, "w", encoding="utf-8") as f:
    json.dump(chunk_metadata, f, ensure_ascii=False, indent=2)

print("Đã ghi:", KB_CHUNKS_PATH)
print("Đã ghi:", CHUNK_METADATA_PATH)

In [ ]:
for chunk in kb_chunks[:3]:
    print("=" * 100)
    print("CHUNK ID:", chunk["chunk_id"])
    print("TITLE   :", chunk["title"])
    print("SECTION :", chunk["section"])
    print("TEXT:\n")
    print(chunk["chunk_text"][:2000])
    print("\n")